<a href="https://colab.research.google.com/github/diyashreesukul/Under-Water-Trash-Detection/blob/main/LearningAHP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Importing the dataset


In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"diyashreesukul","key":"38d3a5ef4a8562f059e1a3eecb16ae27"}'}

In [ ]:
import os

os.makedirs('/root/.kaggle', exist_ok=True)
!mv kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d samuelotiattakorah/agriculture-crop-yield
!unzip agriculture-crop-yield.zip

Dataset URL: https://www.kaggle.com/datasets/samuelotiattakorah/agriculture-crop-yield
License(s): CC0-1.0
100% 33.4M/33.4M [00:02<00:00, 12.9MB/s]

Archive:  agriculture-crop-yield.zip
  inflating: crop_yield.csv          


In [ ]:
import pandas as pd

df = pd.read_csv('crop_yield.csv')   # filename may vary slightly
df.head()

,Region,Soil_Type,Crop,Rainfall_mm,Temperature_Celsius,Fertilizer_Used,Irrigation_Used,Weather_Condition,Days_to_Harvest,Yield_tons_per_hectare
0,West,Sandy,Cotton,897.077239,27.676966,False,True,Cloudy,122,6.555816
1,South,Clay,Rice,992.673282,18.026142,True,True,Rainy,140,8.527341
2,North,Loam,Barley,147.998025,29.794042,False,False,Sunny,106,1.127443
3,North,Sandy,Soybean,986.866331,16.644190,False,True,Rainy,146,6.517573
4,South,Silt,Wheat,730.379174,31.620687,True,True,Cloudy,110,7.248251


Now modifying the dataset as per reqirement


In [ ]:
import numpy as np
import pandas as pd

# =====================================================
# DATASET
# =====================================================

data = {

    "Crop": ["Cotton", "Rice", "Barley", "Soybean", "Wheat"],

    "Region": ["West", "South", "North", "North", "South"],

    "Soil_Type": ["Sandy", "Clay", "Loam", "Silt", "Silt"],

    "Rainfall_mm": [
        897.077239,
        992.673282,
        147.998025,
        986.866331,
        730.379174
    ],

    "Temperature_Celsius": [
        27.676966,
        18.026142,
        29.794042,
        16.644190,
        31.620687
    ],

    "Fertilizer_Used": [
        False,
        True,
        False,
        False,
        True
    ],

    "Irrigation_Used": [
        True,
        True,
        False,
        True,
        True
    ],

    "Weather_Condition": [
        "Cloudy",
        "Rainy",
        "Sunny",
        "Rainy",
        "Cloudy"
    ]
}

df = pd.DataFrame(data)

# =====================================================
# SEASONAL WEIGHTS
# =====================================================

season_weights = {

    "kharif": np.array([
        0.063,   # Region
        0.115,   # Soil
        0.400,   # Rain
        0.202,   # Temp
        0.063,   # Fert
        0.063,   # Irr
        0.115    # Weather
    ]),

    "rabi": np.array([
        0.080,
        0.120,
        0.180,
        0.350,
        0.080,
        0.080,
        0.110
    ]),

    "zaid": np.array([
        0.070,
        0.100,
        0.220,
        0.300,
        0.070,
        0.120,
        0.120
    ])
}

# =====================================================
# SEASON IDEAL TEMPERATURES
# =====================================================

ideal_temp = {

    "kharif": 25,
    "rabi": 18,
    "zaid": 32

}

# =====================================================
# WEATHER MAP
# =====================================================

weather_map = {

    "rainy": 1.0,
    "cloudy": 0.8,
    "sunny": 0.5

}

# =====================================================
# REGION MAP
# =====================================================

region_map = {

    "north": 1.0,
    "south": 0.8,
    "west": 0.6

}

# =====================================================
# SOIL MAP
# =====================================================

soil_map = {

    "clay": 1.0,
    "loam": 0.8,
    "silt": 0.7,
    "sandy": 0.5

}

# =====================================================
# MAIN FUNCTION
# =====================================================

def run_model():

    # -------------------------------------------------
    # INPUTS
    # -------------------------------------------------

    season = input(
        "Enter season (kharif/rabi/zaid): "
    ).lower()

    confidence = float(
        input("Enter confidence level (0-100): ")
    )

    # -------------------------------------------------
    # SELECT WEIGHTS
    # -------------------------------------------------

    weights = season_weights[season]

    # -------------------------------------------------
    # NORMALIZATION VALUES
    # -------------------------------------------------

    max_rain = df["Rainfall_mm"].max()

    temp_target = ideal_temp[season]

    temp_range = 20

    scores = []

    # -------------------------------------------------
    # COMPUTE SCORES
    # -------------------------------------------------

    for _, row in df.iterrows():

        # ---------------------------------------------
        # REGION SCORE
        # ---------------------------------------------

        region_score = region_map[
            row["Region"].lower()
        ]

        # ---------------------------------------------
        # SOIL SCORE
        # ---------------------------------------------

        soil_score = soil_map[
            row["Soil_Type"].lower()
        ]

        # ---------------------------------------------
        # RAIN SCORE
        # ---------------------------------------------

        rain_score = (
            row["Rainfall_mm"] / max_rain
        )

        # ---------------------------------------------
        # TEMP SCORE
        # ---------------------------------------------

        temp_score = 1 - (

            abs(
                row["Temperature_Celsius"]
                - temp_target
            ) / temp_range

        )

        temp_score = max(temp_score, 0)

        # ---------------------------------------------
        # FERTILIZER SCORE
        # ---------------------------------------------

        fert_score = 1 if row[
            "Fertilizer_Used"
        ] else 0

        # ---------------------------------------------
        # IRRIGATION SCORE
        # ---------------------------------------------

        irr_score = 1 if row[
            "Irrigation_Used"
        ] else 0

        # ---------------------------------------------
        # WEATHER SCORE
        # ---------------------------------------------

        weather_score = weather_map[
            row["Weather_Condition"].lower()
        ]

        # ---------------------------------------------
        # DECISION VECTOR
        # ---------------------------------------------

        X = np.array([

            region_score,
            soil_score,
            rain_score,
            temp_score,
            fert_score,
            irr_score,
            weather_score

        ])

        # ---------------------------------------------
        # FINAL SCORE
        # ---------------------------------------------

        final_score = np.dot(
            weights,
            X
        )

        scores.append(final_score)

    # =================================================
    # STORE SCORES
    # =================================================

    df["Final_Score"] = scores

    # =================================================
    # CROP LEVEL AGGREGATION
    # =================================================

    crop_scores = (

        df.groupby("Crop")["Final_Score"]
        .mean()
        .reset_index()

    )

    # =================================================
    # NORMALIZE SCORES
    # =================================================

    max_score = crop_scores[
        "Final_Score"
    ].max()

    crop_scores["Normalized_Score"] = (

        crop_scores["Final_Score"]
        / max_score

    )

    # =================================================
    # SORT DESCENDING
    # =================================================

    crop_scores = crop_scores.sort_values(

        by="Final_Score",
        ascending=False

    )

    # =================================================
    # DISPLAY SCORES
    # =================================================

    print("\n================================")
    print("FINAL CROP PRIORITIES")
    print("================================\n")

    print(crop_scores)

    # =================================================
    # GROUP DECISION
    # =================================================

    threshold = confidence / 100

    final_group = crop_scores[

        crop_scores["Normalized_Score"]
        >= threshold

    ]

    # =================================================
    # DISPLAY GROUP DECISION
    # =================================================

    print("\n================================")
    print("GROUP DECISION")
    print("================================\n")

    print(
        "Confidence Threshold:",
        threshold
    )

    print("\nRecommended Crops:\n")

    if len(final_group) == 0:

        print("No crops satisfy threshold")

    else:

        for _, row in final_group.iterrows():

            print(

                f"{row['Crop']} "
                f"(Priority = "
                f"{round(row['Normalized_Score'],3)})"

            )

# =====================================================
# RUN MODEL
# =====================================================

run_model()

Enter season (kharif/rabi/zaid): Kharif
Enter confidence level (0-100): 80

FINAL CROP PRIORITIES

      Crop  Final_Score  Normalized_Score
2     Rice     0.937964          1.000000
3  Soybean     0.836766          0.892109
1   Cotton     0.786742          0.838776
4    Wheat     0.778339          0.829818
0   Barley     0.425716          0.453873

GROUP DECISION

Confidence Threshold: 0.8

Recommended Crops:

Rice (Priority = 1.0)
Soybean (Priority = 0.892)
Cotton (Priority = 0.839)
Wheat (Priority = 0.83)


In [ ]:
# =====================================================
# TABULAR OUTPUT FOR ALL 3 SEASONS
# USING SAME VARIABLES AS YOUR CODE
# =====================================================

season_results = {}

# =====================================================
# LOOP THROUGH ALL SEASONS
# =====================================================

for season in ["kharif", "rabi", "zaid"]:

    weights = season_weights[season]

    max_rain = df["Rainfall_mm"].max()

    temp_target = ideal_temp[season]

    temp_range = 20

    scores = []

    # -------------------------------------------------
    # CALCULATE SCORES
    # -------------------------------------------------

    for _, row in df.iterrows():

        region_score = region_map[
            row["Region"].lower()
        ]

        soil_score = soil_map[
            row["Soil_Type"].lower()
        ]

        rain_score = (
            row["Rainfall_mm"] / max_rain
        )

        temp_score = 1 - (

            abs(
                row["Temperature_Celsius"]
                - temp_target
            ) / temp_range

        )

        temp_score = max(temp_score, 0)

        fert_score = 1 if row[
            "Fertilizer_Used"
        ] else 0

        irr_score = 1 if row[
            "Irrigation_Used"
        ] else 0

        weather_score = weather_map[
            row["Weather_Condition"].lower()
        ]

        # ---------------------------------------------
        # DECISION VECTOR
        # ---------------------------------------------

        X = np.array([

            region_score,
            soil_score,
            rain_score,
            temp_score,
            fert_score,
            irr_score,
            weather_score

        ])

        # ---------------------------------------------
        # FINAL SCORE
        # ---------------------------------------------

        final_score = np.dot(
            weights,
            X
        )

        scores.append(final_score)

    # =================================================
    # STORE SCORES
    # =================================================

    temp_df = df.copy()

    temp_df["Final_Score"] = scores

    # =================================================
    # CROP LEVEL AGGREGATION
    # =================================================

    crop_scores = (

        temp_df.groupby("Crop")["Final_Score"]
        .mean()
        .reset_index()

    )

    # =================================================
    # NORMALIZE
    # =================================================

    max_score = crop_scores[
        "Final_Score"
    ].max()

    crop_scores[season] = (

        crop_scores["Final_Score"]
        / max_score

    )

    # =================================================
    # STORE RESULTS
    # =================================================

    season_results[season] = crop_scores[
        ["Crop", season]
    ]

# =====================================================
# MERGE ALL SEASON RESULTS
# =====================================================

final_df = season_results["kharif"]

final_df = final_df.merge(
    season_results["rabi"],
    on="Crop"
)

final_df = final_df.merge(
    season_results["zaid"],
    on="Crop"
)

# =====================================================
# GENERATE RECOMMENDED SEASONS
# =====================================================

recommendations = []

for _, row in final_df.iterrows():

    scores = {

        "Kharif": row["kharif"],
        "Rabi": row["rabi"],
        "Zaid": row["zaid"]

    }

    best_score = max(
        scores.values()
    )

    suitable = []

    # 90% RULE

    for season, score in scores.items():

        if score >= 0.90 * best_score:

            suitable.append(season)

    recommendations.append(
        ", ".join(suitable)
    )

# =====================================================
# ADD RECOMMENDATION COLUMN
# =====================================================

final_df[
    "Recommended_Seasons"
] = recommendations

# =====================================================
# SORT
# =====================================================

final_df = final_df.sort_values(

    by="kharif",
    ascending=False

)

# =====================================================
# RESET INDEX
# =====================================================

final_df.reset_index(

    drop=True,
    inplace=True

)

# =====================================================
# DISPLAY FINAL TABLE
# =====================================================

print("\n========================================")
print("SEASONAL CROP RECOMMENDATION TABLE")
print("========================================\n")

print(final_df.round(3))


SEASONAL CROP RECOMMENDATION TABLE

      Crop  kharif   rabi   zaid Recommended_Seasons
0     Rice   1.000  1.000  0.894        Kharif, Rabi
1  Soybean   0.892  0.874  0.770        Kharif, Rabi
2   Cotton   0.839  0.630  0.855        Kharif, Zaid
3    Wheat   0.830  0.651  1.000                Zaid
4   Barley   0.454  0.408  0.587                Zaid


In [ ]:
import numpy as np
import pandas as pd

# =====================================================
# DATASET
# =====================================================

df = pd.read_csv("crop_yield.csv")


# =====================================================
# SEASONAL WEIGHTS
# =====================================================

season_weights = {

    "kharif": np.array([
        0.063,   # Region
        0.115,   # Soil
        0.400,   # Rain
        0.202,   # Temp
        0.063,   # Fert
        0.063,   # Irr
        0.115    # Weather
    ]),

    "rabi": np.array([
        0.080,
        0.120,
        0.180,
        0.350,
        0.080,
        0.080,
        0.110
    ]),

    "zaid": np.array([
        0.070,
        0.100,
        0.220,
        0.300,
        0.070,
        0.120,
        0.120
    ])
}

# =====================================================
# SEASON IDEAL TEMPERATURES
# =====================================================

ideal_temp = {

    "kharif": 25,
    "rabi": 18,
    "zaid": 32

}

# =====================================================
# WEATHER MAP
# =====================================================

weather_map = {

    "rainy": 1.0,
    "cloudy": 0.8,
    "sunny": 0.5

}

# =====================================================
# REGION MAP
# =====================================================

region_map = {

    "north": 1.0,
    "south": 0.8,
    "west": 0.6

}

# =====================================================
# SOIL MAP
# =====================================================

soil_map = {

    "clay": 1.0,
    "loam": 0.8,
    "silt": 0.7,
    "sandy": 0.5

}

# =====================================================
# MAIN FUNCTION
# =====================================================

def run_model():

    # -------------------------------------------------
    # INPUTS
    # -------------------------------------------------

    season = input(
        "Enter season (kharif/rabi/zaid): "
    ).lower()

    confidence = float(
        input("Enter confidence level (0-100): ")
    )

    # -------------------------------------------------
    # SELECT WEIGHTS
    # -------------------------------------------------

    weights = season_weights[season]

    # -------------------------------------------------
    # NORMALIZATION VALUES
    # -------------------------------------------------

    max_rain = df["Rainfall_mm"].max()

    temp_target = ideal_temp[season]

    temp_range = 20

    scores = []

    # -------------------------------------------------
    # COMPUTE SCORES
    # -------------------------------------------------

    for _, row in df.iterrows():

        # ---------------------------------------------
        # REGION SCORE
        # ---------------------------------------------

        region_score = region_map.get(
            row["Region"].lower(), 0.5
        )

        # ---------------------------------------------
        # SOIL SCORE
        # ---------------------------------------------

        soil_score = soil_map.get(
            row["Soil_Type"].lower(), 0.5
        )

        # ---------------------------------------------
        # RAIN SCORE
        # ---------------------------------------------

        rain_score = (
            row["Rainfall_mm"] / max_rain
        )

        # ---------------------------------------------
        # TEMP SCORE
        # ---------------------------------------------

        temp_score = 1 - (

            abs(
                row["Temperature_Celsius"]
                - temp_target
            ) / temp_range

        )

        temp_score = max(temp_score, 0)

        # ---------------------------------------------
        # FERTILIZER SCORE
        # ---------------------------------------------

        fert_score = 1 if row[
            "Fertilizer_Used"
        ] else 0

        # ---------------------------------------------
        # IRRIGATION SCORE
        # ---------------------------------------------

        irr_score = 1 if row[
            "Irrigation_Used"
        ] else 0

        # ---------------------------------------------
        # WEATHER SCORE
        # ---------------------------------------------

        weather_score = weather_map.get(
            row["Weather_Condition"].lower(), 0.5
        )

        # ---------------------------------------------
        # DECISION VECTOR
        # ---------------------------------------------

        X = np.array([

            region_score,
            soil_score,
            rain_score,
            temp_score,
            fert_score,
            irr_score,
            weather_score

        ])

        # ---------------------------------------------
        # FINAL SCORE
        # ---------------------------------------------

        final_score = np.dot(
            weights,
            X
        )

        scores.append(final_score)

    # =================================================
    # STORE SCORES
    # =================================================

    df["Final_Score"] = scores

    # =================================================
    # CROP LEVEL AGGREGATION
    # =================================================

    crop_scores = (

        df.groupby("Crop")["Final_Score"]
        .mean()
        .reset_index()

    )

    # =================================================
    # NORMALIZE SCORES
    # =================================================

    max_score = crop_scores[
        "Final_Score"
    ].max()

    crop_scores["Normalized_Score"] = (

        crop_scores["Final_Score"]
        / max_score

    )

    # =================================================
    # SORT DESCENDING
    # =================================================

    crop_scores = crop_scores.sort_values(

        by="Final_Score",
        ascending=False

    )

    # =================================================
    # DISPLAY SCORES
    # =================================================

    print("\n================================")
    print("FINAL CROP PRIORITIES")
    print("================================\n")

    print(crop_scores)

    # =================================================
    # GROUP DECISION
    # =================================================

    threshold = confidence / 100

    final_group = crop_scores[

        crop_scores["Normalized_Score"]
        >= threshold

    ]

    # =================================================
    # DISPLAY GROUP DECISION
    # =================================================

    print("\n================================")
    print("GROUP DECISION")
    print("================================\n")

    print(
        "Confidence Threshold:",
        threshold
    )

    print("\nRecommended Crops:\n")

    if len(final_group) == 0:

        print("No crops satisfy threshold")

    else:

        for _, row in final_group.iterrows():

            print(

                f"{row['Crop']} "
                f"(Priority = "
                f"{round(row['Normalized_Score'],3)})",

            )

# =====================================================
# RUN MODEL
# =====================================================

run_model()

Enter season (kharif/rabi/zaid): Kharif
Enter confidence level (0-100): 80

FINAL CROP PRIORITIES

      Crop  Final_Score  Normalized_Score
3     Rice     0.630048          1.000000
5    Wheat     0.629977          0.999886
4  Soybean     0.629871          0.999718
0   Barley     0.629705          0.999455
1   Cotton     0.629461          0.999068
2    Maize     0.629416          0.998996

GROUP DECISION

Confidence Threshold: 0.8

Recommended Crops:

Rice (Priority = 1.0)
Wheat (Priority = 1.0)
Soybean (Priority = 1.0)
Barley (Priority = 0.999)
Cotton (Priority = 0.999)
Maize (Priority = 0.999)


In [ ]:
# dynamic alpha
import numpy as np
import pandas as pd

# =====================================================
# LOAD DATASET
# =====================================================

df = pd.read_csv("crop_yield.csv")

# =====================================================
# AHP WEIGHTS
# =====================================================

season_weights = {

    "kharif": np.array([
        0.063,
        0.115,
        0.400,
        0.202,
        0.063,
        0.063,
        0.115
    ]),

    "rabi": np.array([
        0.080,
        0.120,
        0.180,
        0.350,
        0.080,
        0.080,
        0.110
    ]),

    "zaid": np.array([
        0.070,
        0.100,
        0.220,
        0.300,
        0.070,
        0.120,
        0.120
    ])
}

# =====================================================
# ANP DEPENDENCY MATRICES
# =====================================================

dependency_matrices = {

    "kharif": np.array([

        [1,0,0,0,0,0,0],
        [0,1,0.2,0,0,0.1,0],
        [0,0.1,1,0,0,0.2,0.2],
        [0,0,0,1,0,0,0.1],
        [0,0,0,0,1,0,0],
        [0,0.1,0.2,0,0,1,0],
        [0,0,0.1,0.1,0,0,1]

    ]),

    "rabi": np.array([

        [1,0,0,0,0,0,0],
        [0,1,0.1,0,0,0.1,0],
        [0,0.1,1,0.1,0,0.1,0.1],
        [0,0,0.2,1,0,0.2,0.2],
        [0,0,0,0,1,0,0],
        [0,0.1,0.1,0.1,0,1,0],
        [0,0,0.1,0.2,0,0.1,1]

    ]),

    "zaid": np.array([

        [1,0,0,0,0,0,0],
        [0,1,0.1,0,0,0.1,0],
        [0,0.1,1,0.1,0,0.2,0.1],
        [0,0,0.2,1,0,0.2,0.1],
        [0,0,0,0,1,0,0],
        [0,0.1,0.1,0.2,0,1,0.1],
        [0,0,0.1,0.1,0,0.1,1]

    ])
}

# =====================================================
# IDEAL TEMPERATURES
# =====================================================

ideal_temp = {

    "kharif": 25,
    "rabi": 18,
    "zaid": 32
}

# =====================================================
# WEATHER MAP
# =====================================================

weather_map = {

    "rainy": 1.0,
    "cloudy": 0.8,
    "sunny": 0.5
}

# =====================================================
# REGION MAP
# =====================================================

region_map = {

    "north": 1.0,
    "south": 0.8,
    "east": 0.7,
    "west": 0.6
}

# =====================================================
# SOIL MAP
# =====================================================

soil_map = {

    "clay": 1.0,
    "loam": 0.8,
    "silt": 0.7,
    "sandy": 0.5
}

# =====================================================
# ALPHA
# =====================================================

def compute_alpha(B):

    n = B.shape[0]

    off_diag_sum = np.sum(B) - np.trace(B)

    off_diag_count = n*n - n

    return off_diag_sum / off_diag_count

# =====================================================
# ANP WEIGHTS
# =====================================================

def compute_anp_weights(season):

    W = season_weights[season]

    B = dependency_matrices[season]

    alpha = compute_alpha(B)

    BW = np.dot(B, W)

    Wanp = (1 - alpha) * W + alpha * BW

    Wanp = Wanp / np.sum(Wanp)

    return Wanp

# =====================================================
# INPUTS
# =====================================================

season = input(
    "Enter season (kharif/rabi/zaid): "
).lower()

confidence = float(
    input(
        "Enter confidence level (0-100): "
    )
)

# =====================================================
# ANP WEIGHTS
# =====================================================

weights = compute_anp_weights(season)

print("\nANP Weights:\n")

criteria = [
    "Region",
    "Soil",
    "Rain",
    "Temp",
    "Fert",
    "Irr",
    "Weather"
]

for c, w in zip(criteria, weights):

    print(
        f"{c:<10} : {round(w,4)}"
    )

# =====================================================
# NORMALIZATION VALUES
# =====================================================

max_rain = df["Rainfall_mm"].max()

temp_target = ideal_temp[season]

temp_range = 20

scores = []

# =====================================================
# COMPUTE SCORES
# =====================================================

for _, row in df.iterrows():

    region_score = region_map.get(
        str(row["Region"]).lower(),
        0.5
    )

    soil_score = soil_map.get(
        str(row["Soil_Type"]).lower(),
        0.5
    )

    rain_score = (
        row["Rainfall_mm"]
        / max_rain
    )

    temp_score = 1 - (

        abs(
            row["Temperature_Celsius"]
            - temp_target
        ) / temp_range

    )

    temp_score = max(
        temp_score,
        0
    )

    fert_score = 1 if row[
        "Fertilizer_Used"
    ] else 0

    irr_score = 1 if row[
        "Irrigation_Used"
    ] else 0

    weather_score = weather_map.get(

        str(
            row["Weather_Condition"]
        ).lower(),

        0.5
    )

    X = np.array([

        region_score,
        soil_score,
        rain_score,
        temp_score,
        fert_score,
        irr_score,
        weather_score

    ])

    final_score = np.dot(
        weights,
        X
    )

    scores.append(
        final_score
    )

# =====================================================
# STORE SCORES
# =====================================================

df["Final_Score"] = scores

# =====================================================
# CROP AGGREGATION
# =====================================================

crop_scores = (

    df.groupby("Crop")
    ["Final_Score"]
    .mean()
    .reset_index()

)

# =====================================================
# NORMALIZE
# =====================================================

max_score = crop_scores["Final_Score"].max()

crop_scores[season] = (
    crop_scores["Final_Score"] / max_score
)

# Contrast enhancement
crop_scores[season] = crop_scores[season] ** 2
# =====================================================
# SORT
# =====================================================

crop_scores = crop_scores.sort_values(

    by="Final_Score",

    ascending=False

)

# =====================================================
# DISPLAY SCORES
# =====================================================

print("\n================================")
print("FINAL CROP PRIORITIES")
print("================================\n")

print(
    crop_scores.round(4)
)

# =====================================================
# GROUP DECISION
# =====================================================

threshold = confidence / 100

final_group = crop_scores[

    crop_scores[
        season
    ] >= threshold

]

# =====================================================
# DISPLAY GROUP
# =====================================================

print("\n================================")
print("GROUP DECISION")
print("================================\n")

print(
    "Confidence Threshold:",
    threshold
)

print("\nRecommended Crops:\n")

if len(final_group) == 0:

    print(
        "No crops satisfy threshold"
    )

else:

    for _, row in final_group.iterrows():

        print(

            f"{row['Crop']} "
            f"(Priority = "
            f"{round(row[season],3)})"

        )

Enter season (kharif/rabi/zaid): kharif
Enter confidence level (0-100): 80

ANP Weights:

Region     : 0.0611
Soil       : 0.1143
Rain       : 0.3895
Temp       : 0.1963
Fert       : 0.0611
Irr        : 0.0641
Weather    : 0.1135

FINAL CROP PRIORITIES

      Crop  Final_Score  kharif
3     Rice       0.6201  1.0000
5    Wheat       0.6200  0.9997
4  Soybean       0.6200  0.9996
0   Barley       0.6198  0.9990
1   Cotton       0.6196  0.9983
2    Maize       0.6195  0.9980

GROUP DECISION

Confidence Threshold: 0.8

Recommended Crops:

Rice (Priority = 1.0)
Wheat (Priority = 1.0)
Soybean (Priority = 1.0)
Barley (Priority = 0.999)
Cotton (Priority = 0.998)
Maize (Priority = 0.998)


In [ ]:
import pandas as pd
import numpy as np

# =====================================================
# LOAD DATASET
# =====================================================

df = pd.read_csv("crop_yield.csv")

# =====================================================
# AHP WEIGHTS
# Region, Soil, Rain, Temp,
# Fert, Irr, Weather
# =====================================================

season_weights = {

    "kharif": np.array([
        0.063,
        0.115,
        0.400,
        0.202,
        0.063,
        0.063,
        0.115
    ]),

    "rabi": np.array([
        0.080,
        0.120,
        0.180,
        0.350,
        0.080,
        0.080,
        0.110
    ]),

    "zaid": np.array([
        0.070,
        0.100,
        0.220,
        0.300,
        0.070,
        0.120,
        0.120
    ])
}

# =====================================================
# ANP DEPENDENCY MATRICES
# =====================================================

dependency_matrices = {

    "kharif": np.array([

        [1,0,0,0,0,0,0],
        [0,1,0.2,0,0,0.1,0],
        [0,0.1,1,0,0,0.2,0.2],
        [0,0,0,1,0,0,0.1],
        [0,0,0,0,1,0,0],
        [0,0.1,0.2,0,0,1,0],
        [0,0,0.1,0.1,0,0,1]

    ]),

    "rabi": np.array([

        [1,0,0,0,0,0,0],
        [0,1,0.1,0,0.1,0.1,0],
        [0,0.1,1,0.1,0,0.1,0.1],
        [0,0,0.1,1,0,0.2,0.2],
        [0,0.1,0,0,1,0,0],
        [0,0.1,0.1,0.2,0,1,0],
        [0,0,0.1,0.2,0,0,1]

    ]),

    "zaid": np.array([

        [1,0,0,0,0,0,0],
        [0,1,0.1,0,0,0.1,0],
        [0,0.1,1,0.1,0,0.1,0.1],
        [0,0,0.1,1,0,0.2,0.1],
        [0,0,0,0,1,0,0],
        [0,0.1,0.1,0.2,0,1,0.1],
        [0,0,0.1,0.1,0,0.1,1]

    ])
}

# =====================================================
# ANP WEIGHT FUNCTION
# W_ANP = W + alpha(BW)
# =====================================================

def compute_anp_weights(season):

    W = season_weights[season]

    B = dependency_matrices[season]

    alpha = 0.3

    BW = np.dot(B, W)

    Wanp = W + alpha * BW

    Wanp = Wanp / np.sum(Wanp)

    return Wanp

# =====================================================
# IDEAL TEMPERATURE
# =====================================================

ideal_temp = {

    "kharif": 25,
    "rabi": 18,
    "zaid": 32

}

# =====================================================
# WEATHER MAP
# =====================================================

weather_map = {

    "rainy": 1.0,
    "cloudy": 0.8,
    "sunny": 0.5

}

# =====================================================
# REGION MAP
# =====================================================

region_map = {

    "north": 1.0,
    "south": 0.8,
    "west": 0.6,
    "east": 0.7

}

# =====================================================
# SOIL MAP
# =====================================================

soil_map = {

    "clay": 1.0,
    "loam": 0.8,
    "silt": 0.7,
    "sandy": 0.5

}

# =====================================================
# STORE RESULTS
# =====================================================

season_results = {}

# =====================================================
# PROCESS EACH SEASON
# =====================================================

for season in ["kharif", "rabi", "zaid"]:

    weights = compute_anp_weights(season)

    max_rain = df["Rainfall_mm"].max()

    temp_target = ideal_temp[season]

    temp_range = 20

    scores = []

    for _, row in df.iterrows():

        region_score = region_map.get(
            str(row["Region"]).lower(),
            0.5
        )

        soil_score = soil_map.get(
            str(row["Soil_Type"]).lower(),
            0.5
        )

        rain_score = (
            row["Rainfall_mm"] / max_rain
        )

        temp_score = 1 - (

            abs(
                row["Temperature_Celsius"]
                - temp_target
            ) / temp_range

        )

        temp_score = max(temp_score, 0)

        fert_score = (
            1 if row["Fertilizer_Used"]
            else 0
        )

        irr_score = (
            1 if row["Irrigation_Used"]
            else 0
        )

        weather_score = weather_map.get(
            str(
                row["Weather_Condition"]
            ).lower(),
            0.5
        )

        X = np.array([

            region_score,
            soil_score,
            rain_score,
            temp_score,
            fert_score,
            irr_score,
            weather_score

        ])

        final_score = np.dot(
            weights,
            X
        )

        scores.append(final_score)

    # =================================================
    # STORE SCORES
    # =================================================

    temp_df = df.copy()

    temp_df["Final_Score"] = scores

    # =================================================
    # CROP AGGREGATION
    # =================================================

    crop_scores = (

        temp_df.groupby("Crop")
        ["Final_Score"]
        .mean()
        .reset_index()

    )

    # =================================================
    # NORMALIZATION
    # =================================================

    max_score = crop_scores[
        "Final_Score"
    ].max()

    crop_scores[season] = (

        crop_scores["Final_Score"]
        / max_score

    )

    season_results[season] = crop_scores[
        ["Crop", season]
    ]

# =====================================================
# MERGE RESULTS
# =====================================================

final_df = season_results["kharif"]

final_df = final_df.merge(
    season_results["rabi"],
    on="Crop"
)

final_df = final_df.merge(
    season_results["zaid"],
    on="Crop"
)

# =====================================================
# RECOMMENDED SEASONS
# =====================================================

recommendations = []

for _, row in final_df.iterrows():

    scores = {

        "Kharif": row["kharif"],
        "Rabi": row["rabi"],
        "Zaid": row["zaid"]

    }

    best_score = max(
        scores.values()
    )

    suitable = []

    for season_name, score in scores.items():

        if score >= 0.70 * best_score:

            suitable.append(
                season_name
            )

    recommendations.append(
        ", ".join(suitable)
    )

final_df[
    "Recommended_Seasons"
] = recommendations

# =====================================================
# SORT
# =====================================================

final_df = final_df.sort_values(

    by="kharif",
    ascending=False

)

final_df.reset_index(

    drop=True,
    inplace=True

)

# =====================================================
# DISPLAY
# =====================================================

print("\n===================================")
print("ANP SEASONAL RECOMMENDATION TABLE")
print("===================================\n")

print(final_df.round(3))


ANP SEASONAL RECOMMENDATION TABLE

      Crop  kharif   rabi   zaid Recommended_Seasons
0     Rice   1.000  1.000  0.999  Kharif, Rabi, Zaid
1    Wheat   1.000  1.000  1.000  Kharif, Rabi, Zaid
2  Soybean   1.000  1.000  1.000  Kharif, Rabi, Zaid
3   Barley   1.000  1.000  0.999  Kharif, Rabi, Zaid
4   Cotton   0.999  0.999  0.999  Kharif, Rabi, Zaid
5    Maize   0.999  1.000  0.999  Kharif, Rabi, Zaid


In [ ]:
# dynamic alpha
import numpy as np
import pandas as pd

# =====================================================
# LOAD DATASET
# =====================================================

df = pd.read_csv("crop_yield.csv")

# =====================================================
# AHP WEIGHTS
# =====================================================

season_weights = {

    "kharif": np.array([
        0.063,
        0.115,
        0.400,
        0.202,
        0.063,
        0.063,
        0.115
    ]),

    "rabi": np.array([
        0.080,
        0.120,
        0.180,
        0.350,
        0.080,
        0.080,
        0.110
    ]),

    "zaid": np.array([
        0.070,
        0.100,
        0.220,
        0.300,
        0.070,
        0.120,
        0.120
    ])
}

# =====================================================
# ANP DEPENDENCY MATRICES
# =====================================================

dependency_matrices = {

    "kharif": np.array([

        [1,0,0,0,0,0,0],
        [0,1,0.2,0,0,0.1,0],
        [0,0.1,1,0,0,0.2,0.2],
        [0,0,0,1,0,0,0.1],
        [0,0,0,0,1,0,0],
        [0,0.1,0.2,0,0,1,0],
        [0,0,0.1,0.1,0,0,1]

    ]),

    "rabi": np.array([

        [1,0,0,0,0,0,0],
        [0,1,0.1,0,0,0.1,0],
        [0,0.1,1,0.1,0,0.1,0.1],
        [0,0,0.2,1,0,0.2,0.2],
        [0,0,0,0,1,0,0],
        [0,0.1,0.1,0.1,0,1,0],
        [0,0,0.1,0.2,0,0.1,1]

    ]),

    "zaid": np.array([

        [1,0,0,0,0,0,0],
        [0,1,0.1,0,0,0.1,0],
        [0,0.1,1,0.1,0,0.2,0.1],
        [0,0,0.2,1,0,0.2,0.1],
        [0,0,0,0,1,0,0],
        [0,0.1,0.1,0.2,0,1,0.1],
        [0,0,0.1,0.1,0,0.1,1]

    ])
}

# =====================================================
# IDEAL TEMPERATURES
# =====================================================

ideal_temp = {

    "kharif": 25,
    "rabi": 18,
    "zaid": 32
}

# =====================================================
# WEATHER MAP
# =====================================================

weather_scores = {

    "kharif": {
        "rainy":1.0,
        "cloudy":0.8,
        "sunny":0.2
    },

    "rabi": {
        "sunny":1.0,
        "cloudy":0.7,
        "rainy":0.4
    },

    "zaid": {
        "sunny":1.0,
        "cloudy":0.6,
        "rainy":0.2
    }

}
# =====================================================
# REGION MAP
# =====================================================

region_map = {

    "north":1.00,
    "south":0.75,
    "east":0.55,
    "west":0.35

}

# =====================================================
# SOIL MAP
# =====================================================

soil_map = {

    "clay":1.00,
    "loam":0.85,
    "silt":0.60,
    "sandy":0.35

}

# =====================================================
# ALPHA
# =====================================================

def compute_alpha(B):

    n = B.shape[0]

    off_diag_sum = np.sum(B) - np.trace(B)

    off_diag_count = n*n - n

    return off_diag_sum / off_diag_count

# =====================================================
# ANP WEIGHTS
# =====================================================

def compute_anp_weights(season):

    W = season_weights[season]

    B = dependency_matrices[season]

    alpha = compute_alpha(B)

    BW = np.dot(B, W)

    Wanp = (1 - alpha) * W + alpha * BW

    Wanp = Wanp / np.sum(Wanp)

    return Wanp

# =====================================================
# INPUTS
# =====================================================

season = input(
    "Enter season (kharif/rabi/zaid): "
).lower()

confidence = float(
    input(
        "Enter confidence level (0-100): "
    )
)

# =====================================================
# ANP WEIGHTS
# =====================================================

weights = compute_anp_weights(season)

print("\nANP Weights:\n")

criteria = [
    "Region",
    "Soil",
    "Rain",
    "Temp",
    "Fert",
    "Irr",
    "Weather"
]

for c, w in zip(criteria, weights):

    print(
        f"{c:<10} : {round(w,4)}"
    )

# =====================================================
# NORMALIZATION VALUES
# =====================================================

max_rain = df["Rainfall_mm"].max()

temp_target = ideal_temp[season]

temp_range = 10

scores = []

# =====================================================
# COMPUTE SCORES
# =====================================================

for _, row in df.iterrows():

    region_score = region_map.get(
        str(row["Region"]).lower(),
        0.5
    )

    soil_score = soil_map.get(
        str(row["Soil_Type"]).lower(),
        0.5
    )
    ideal_rain = {
    "kharif": 900,
    "rabi": 250,
    "zaid": 600
}

    rain_score = max(
    0,
    1 - abs(
        row["Rainfall_mm"] - ideal_rain[season]
    ) / ideal_rain[season]
  )

    temp_score = 1 - (

        abs(
            row["Temperature_Celsius"]
            - temp_target
        ) / temp_range

    )

    temp_score = max(
        temp_score,
        0
    )

    fert_score = 1 if row[
        "Fertilizer_Used"
    ] else 0

    irr_score = 1 if row[
        "Irrigation_Used"
    ] else 0

    weather_score = weather_scores[season].get(
    str(row["Weather_Condition"]).lower(),
    0.5
)

    X = np.array([

        region_score,
        soil_score,
        rain_score,
        temp_score,
        fert_score,
        irr_score,
        weather_score

    ])

    final_score = np.dot(
        weights,
        X
    )

    scores.append(
        final_score
    )

# =====================================================
# STORE SCORES
# =====================================================

df["Final_Score"] = scores

# =====================================================
# CROP AGGREGATION
# =====================================================

crop_scores = (

    df.groupby("Crop")
    ["Final_Score"]
    .mean()
    .reset_index()

)

# =====================================================
# NORMALIZE
# =====================================================

minimum = crop_scores["Final_Score"].min()
maximum = crop_scores["Final_Score"].max()

crop_scores[season] = (
    crop_scores["Final_Score"]-minimum
)/(maximum-minimum)

# Contrast enhancement
#crop_scores[season] = crop_scores[season] ** 2
# =====================================================
# SORT
# =====================================================

crop_scores = crop_scores.sort_values(

    by="Final_Score",

    ascending=False

)

# =====================================================
# DISPLAY SCORES
# =====================================================

print("\n================================")
print("FINAL CROP PRIORITIES")
print("================================\n")

print(
    crop_scores.round(4)
)

# =====================================================
# GROUP DECISION
# =====================================================

threshold = confidence / 100

final_group = crop_scores[

    crop_scores[
        season
    ] >= threshold

]

# =====================================================
# DISPLAY GROUP
# =====================================================

print("\n================================")
print("GROUP DECISION")
print("================================\n")

print(
    "Confidence Threshold:",
    threshold
)

print("\nRecommended Crops:\n")

if len(final_group) == 0:

    print(
        "No crops satisfy threshold"
    )

else:

    for _, row in final_group.iterrows():

        print(

            f"{row['Crop']} "
            f"(Priority = "
            f"{round(row[season],3)})"

        )

Enter season (kharif/rabi/zaid): kharif
Enter confidence level (0-100): 70

ANP Weights:

Region     : 0.0611
Soil       : 0.1143
Rain       : 0.3895
Temp       : 0.1963
Fert       : 0.0611
Irr        : 0.0641
Weather    : 0.1135

FINAL CROP PRIORITIES

      Crop  Final_Score  kharif
3     Rice       0.5631  1.0000
5    Wheat       0.5630  0.8716
4  Soybean       0.5629  0.7979
0   Barley       0.5629  0.7225
2    Maize       0.5624  0.1383
1   Cotton       0.5623  0.0000

GROUP DECISION

Confidence Threshold: 0.7

Recommended Crops:

Rice (Priority = 1.0)
Wheat (Priority = 0.872)
Soybean (Priority = 0.798)
Barley (Priority = 0.723)
